In [43]:
from langchain_core.messages import content
from langchain_cerebras import ChatCerebras
from langchain_groq import ChatGroq
from langchain import chat_models
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["CEREBRAS_API_KEY"]=os.getenv("CEREBRAS_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


llm_gpt= ChatCerebras( model="gpt-oss-120b",
    api_key=os.getenv("CEREBRAS_API_KEY"))

llm_groq=ChatGroq(model="llama-3.1-8b-instant",
api_key=os.getenv("GROQ_API_KEY"))

#response_gpt =llm_gpt.invoke("Hi mate!")
#response_groq =llm_groq.invoke("Hi mate!")
#print(response_gpt.content)
#print(response_groq.content)

In [44]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt=ChatPromptTemplate.from_messages([
    ("system","You are a fan boy of {character}"),
    ("human","hey dude, create detailed para in {words_count} words about {character} on this comeback from rockbottom to success!")
])

chain_gpt= prompt | llm_gpt
chain_groq=prompt | llm_groq

#for chunk in chain.stream({"character":"batman","words_count":"100" }):
 #   print(chunk.content,end="", flush=True)

batch_inputs =[
{"character":"batman","words_count":"100" },
{"character":"walter white","words_count":"100"} ,
]

#responses = chain_groq.batch(batch_inputs)
#for response in responses:
#    print(response.content)

In [47]:
from langchain_core.tools import tool

@tool
def sendMail(sender:str,receiver:str,sub:str,body:str)->str:
    """Send the mail to the receiver """
    return f"Mail sent successfully to {receiver}"
    

llm_with_tools = llm_groq.bind_tools([sendMail])

message=[{ "role":"user","content":"sender:fightclub@gmail.com , receiver:itslokeshx@gmail.com,sub:dont talk about fight club ,body: invite him to fight club by tyler durden.make the mail so engaging it as welcome to fight club at first line follow with then top 2 rules of it"}]

AI_return_response = llm_with_tools.invoke(message)
AI_return_response.tool_calls

message.append(AI_return_response)

for tool_call in AI_return_response.tool_calls:
    tool_result = sendMail.invoke(tool_call)
    message.append(tool_result)

result=llm_with_tools.invoke(message)
result.content

"Please note that the mail content is fictional and for entertainment purposes only. \n\nAlso, since this is a fictional scenario, I'm assuming the email and sender IDs are valid for this purpose. \n\nIf you want to send real emails, you should use a real email service and replace the email IDs with actual ones."